# LDAPS 결측치 분석 & LDAPS, GFS 파생변수 생성 함수
- ldaps_test에서 결측치 발견 -> Linear Interpolation
- LDAPS, GFS 데이터에서 거리 기반 이론적 power proxy를 정의하고 파생변수로 생성
- 1시간 별로 집계하기 위해 feature별 grid_id의 대표값 설정
- 위 모든 것을 함수로 구현

# 라이브러리

In [1]:
import pandas as pd
import numpy as np

# 데이터 불러오기

In [2]:
ldaps_train = pd.read_csv('../../open/train/ldaps_train.csv', encoding = 'utf-8-sig')
gfs_train = pd.read_csv('../../open/train/gfs_train.csv', encoding = 'utf-8-sig')

ldaps_test = pd.read_csv('../../open/test/ldaps_test.csv', encoding = 'utf-8-sig')
gfs_test = pd.read_csv('../../open/test/gfs_test.csv', encoding = 'utf-8-sig')

# 결측치 확인

In [3]:
ldaps_train.isnull().sum()

forecast_kst_dtm                0
data_available_kst_dtm          0
grid_id                         0
latitude                        0
longitude                       0
heightAboveGround_10_10u        0
heightAboveGround_10_10v        0
heightAboveGround_50_50MUmax    0
heightAboveGround_50_50MUmin    0
heightAboveGround_50_50MVmax    0
heightAboveGround_50_50MVmin    0
heightAboveGround_5_XBLWS       0
heightAboveGround_5_YBLWS       0
heightAboveGround_2_t           0
heightAboveGround_2_dpt         0
heightAboveGround_2_r           0
heightAboveGround_2_q           0
surface_0_sp                    0
meanSea_0_prmsl                 0
etc_0_blh                       0
surface_0_NDNSW                 0
surface_0_NDNLW                 0
heightAboveGround_2_SWDIR       0
heightAboveGround_2_SWDIF       0
etc_0_hcc                       0
etc_0_mcc                       0
etc_0_lcc                       0
etc_0_VLCDC                     0
surface_0_avg_lsprate           0
surface_0_lssr

In [4]:
gfs_train.isnull().sum()

forecast_kst_dtm                  0
data_available_kst_dtm            0
grid_id                           0
latitude                          0
longitude                         0
heightAboveGround_10_10u          0
heightAboveGround_10_10v          0
heightAboveGround_80_u            0
heightAboveGround_80_v            0
heightAboveGround_100_100u        0
heightAboveGround_100_100v        0
heightAboveGround_2_2t            0
heightAboveGround_2_2d            0
heightAboveGround_2_2r            0
heightAboveGround_2_2sh           0
planetaryBoundaryLayer_0_u        0
planetaryBoundaryLayer_0_v        0
planetaryBoundaryLayer_0_VRATE    0
surface_0_dswrf                   0
surface_0_dlwrf                   0
surface_0_prate                   0
surface_0_tp                      0
surface_0_sp                      0
meanSea_0_prmsl                   0
surface_0_gust                    0
lowCloudLayer_0_lcc               0
middleCloudLayer_0_mcc            0
highCloudLayer_0_hcc        

In [5]:
ldaps_test.isnull().sum()

forecast_kst_dtm                 0
data_available_kst_dtm           0
grid_id                          0
latitude                         0
longitude                        0
heightAboveGround_10_10u         0
heightAboveGround_10_10v         0
heightAboveGround_50_50MUmax    48
heightAboveGround_50_50MUmin    48
heightAboveGround_50_50MVmax    48
heightAboveGround_50_50MVmin    48
heightAboveGround_5_XBLWS        0
heightAboveGround_5_YBLWS       16
heightAboveGround_2_t           16
heightAboveGround_2_dpt         16
heightAboveGround_2_r           16
heightAboveGround_2_q           16
surface_0_sp                    48
meanSea_0_prmsl                 48
etc_0_blh                       48
surface_0_NDNSW                  0
surface_0_NDNLW                  0
heightAboveGround_2_SWDIR        0
heightAboveGround_2_SWDIF        0
etc_0_hcc                       48
etc_0_mcc                       48
etc_0_lcc                       48
etc_0_VLCDC                     48
surface_0_avg_lsprat

In [6]:
gfs_test.isnull().sum()

forecast_kst_dtm                  0
data_available_kst_dtm            0
grid_id                           0
latitude                          0
longitude                         0
heightAboveGround_10_10u          0
heightAboveGround_10_10v          0
heightAboveGround_80_u            0
heightAboveGround_80_v            0
heightAboveGround_100_100u        0
heightAboveGround_100_100v        0
heightAboveGround_2_2t            0
heightAboveGround_2_2d            0
heightAboveGround_2_2r            0
heightAboveGround_2_2sh           0
planetaryBoundaryLayer_0_u        0
planetaryBoundaryLayer_0_v        0
planetaryBoundaryLayer_0_VRATE    0
surface_0_dswrf                   0
surface_0_dlwrf                   0
surface_0_prate                   0
surface_0_tp                      0
surface_0_sp                      0
meanSea_0_prmsl                   0
surface_0_gust                    0
lowCloudLayer_0_lcc               0
middleCloudLayer_0_mcc            0
highCloudLayer_0_hcc        

### ldaps_test에만 결측치 발견 -> 중요한 변수들이니 보간해야함
- 2025-04-08 17:00:00
- 2025-06-18 18:00:00
- 2025-07-18 06:00:00
위 시간대만 결측이니 Linear Interpolation -> 함수에 적용시켜놨음

# 결측치 보간 및 파생변수 생성 함수
- LDAPS, GFS
- power proxy 버전

In [7]:
# LDAPS
def preprocess_ldaps_group(
        ldaps_path: str,
        kpx_group: int,
        k: int = 4,
        cp: float = 0.35,
        info_path: str = "/Users/hyukjin/Documents/Gongmo/BARAM-2026/data/info_include_lat_lon.csv"
) -> pd.DataFrame:
    # 물리 상수
    REF_LOW, REF_HIGH, REF_T = 10, 50, 2
    LAPSE_RATE, G, R_D = 0.0065, 9.80665, 287.05

    # 0) 데이터 로드 및 터빈 제원
    ldaps = pd.read_csv(ldaps_path, encoding = 'utf-8-sig')
    info = pd.read_csv(info_path, encoding = 'utf-8-sig')
    group_info = info[info['KPX그룹'] == kpx_group].copy()

    hub_height = float(group_info['Hub Height(m)'].iloc[0])
    rotor_d = float(group_info['Rotor Diameter(m)'].iloc[0])
    swept_area = np.pi * (rotor_d / 2) **2

    # 1) 결측치 보간
    # 같은 예보 배치 안에서만 보간 -> 미래 배치 정보 유입(누수) 차단
    ldaps = ldaps.sort_values(['grid_id', 'data_available_kst_dtm', 'forecast_kst_dtm']).reset_index(drop = True)
    num_cols = [c for c in ldaps.select_dtypes(include = [np.number]).columns if c != 'grid_id']
    ldaps[num_cols] = (
        ldaps.groupby(['grid_id', 'data_available_kst_dtm'])[num_cols]
        .transform(lambda g: g.interpolate(method = 'linear', limit_direction = 'both'))
    )

    # 2) 터빈-격자 최근접 매핑
    grids = ldaps.drop_duplicates('grid_id').sort_values('grid_id').reset_index(drop = True)

    def haversine_km(lat1, lon1, lat2, lon2):
        R = 6371.0088
        lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
        a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
        return 2 * R * np.arcsin(np.sqrt(a))

    grid_lat, grid_lon = grids['latitude'].to_numpy(), grids['longitude'].to_numpy()
    grid_ids = grids['grid_id'].to_numpy()

    records = []
    for _, t in group_info.iterrows():
        dist = haversine_km(t['latitude'], t['longitude'], grid_lat, grid_lon)
        for rank, gi in enumerate(np.argsort(dist)[:k], start = 1):
            records.append({'호기': t['호기'], 'rank': rank,
                            'grid_id': grid_ids[gi], 'dist_km': dist[gi]})
    nearest_info = pd.DataFrame(records)

    # 3) 격자별 파생변수
    ldaps['ws10_ldaps'] = np.hypot(ldaps['heightAboveGround_10_10u'], ldaps['heightAboveGround_10_10v'])
    u50 = (ldaps['heightAboveGround_50_50MUmax'] + ldaps['heightAboveGround_50_50MUmin']) / 2
    v50 = (ldaps['heightAboveGround_50_50MVmax'] + ldaps['heightAboveGround_50_50MVmin']) / 2
    ldaps['ws50_ldaps'] = np.hypot(u50, v50)

    ws10_safe = np.clip(ldaps['ws10_ldaps'], 0.5, None)
    ws50_safe = np.clip(ldaps['ws50_ldaps'], 0.5, None)
    ldaps['alpha'] = np.log(ws50_safe / ws10_safe) / np.log(REF_HIGH / REF_LOW)
    ldaps['alpha'] = ldaps['alpha'].clip(0.05, 0.4)
    ldaps['ws_hub_ldaps'] = ldaps['ws50_ldaps'] * (hub_height / REF_HIGH) ** ldaps['alpha']

    ldaps['t_hub_ldaps'] = ldaps['heightAboveGround_2_t'] - LAPSE_RATE * (hub_height - REF_T)

    q = ldaps['heightAboveGround_2_q']
    tv_sfc = ldaps['heightAboveGround_2_t'] * (1 + 0.61 * q)
    tv_hub = ldaps['t_hub_ldaps'] * (1 + 0.61 * q)
    tv_mean = (tv_sfc + tv_hub) / 2
    p_hub = ldaps['surface_0_sp'] * np.exp(-G * hub_height / (R_D * tv_mean))
    ldaps['rho_hub_ldaps'] = p_hub / (R_D * tv_hub)

    ldaps['power_proxy_ldaps'] = (
        0.5 * ldaps['rho_hub_ldaps'] * swept_area * ldaps['ws_hub_ldaps'] ** 3 * cp / 1000
    )
    ldaps['gust50_ldaps'] = np.hypot(
        ldaps['heightAboveGround_50_50MUmax'] - ldaps['heightAboveGround_50_50MUmin'],
        ldaps['heightAboveGround_50_50MVmax'] - ldaps['heightAboveGround_50_50MVmin'],
    )

    # 4) 터빈별 IDW 가중치
    weight = nearest_info[['호기', 'grid_id', 'dist_km']].copy()
    weight['weight'] = 1 / weight['dist_km']
    weight['weight'] = weight['weight'] / weight.groupby('호기')['weight'].transform('sum')

    # 5) 터빈 단위 축약
    IDW_COLS = [
        'power_proxy_ldaps',
        'ws_hub_ldaps', 'ws10_ldaps', 'ws50_ldaps', 'alpha', 'gust50_ldaps',
        'heightAboveGround_10_10u', 'heightAboveGround_10_10v',
        'rho_hub_ldaps', 't_hub_ldaps',
        'heightAboveGround_2_r', 'heightAboveGround_2_q',
        'surface_0_sp', 'etc_0_blh',
        'surface_0_NDNSW', 'surface_0_NDNLW',
        'etc_0_hcc', 'etc_0_mcc', 'etc_0_lcc',
        'surface_0_lssrate', 'surface_0_SNOM',
        'heightAboveGround_5_XBLWS', 'heightAboveGround_5_YBLWS',
    ]
    merged = ldaps[['forecast_kst_dtm', 'grid_id'] + IDW_COLS].merge(
        weight[['호기', 'grid_id', 'weight']], on = 'grid_id', how = 'inner'
    )
    for col in IDW_COLS:
        merged[col] = merged[col] * merged['weight']
    turbine_level = merged.groupby(['forecast_kst_dtm', '호기'])[IDW_COLS].sum()

    # 6) 그룹 단위 축약
    MEAN_COLS = [c for c in IDW_COLS if c != 'power_proxy_ldaps']
    agg_spec = {c: 'mean' for c in MEAN_COLS}
    agg_spec['power_proxy_ldaps'] = 'sum'

    grouped = turbine_level.groupby('forecast_kst_dtm')
    out = grouped.agg(agg_spec)
    out['ws_hub_std_ldaps'] = grouped['ws_hub_ldaps'].std()
    out = out.rename(columns = {'power_proxy_ldaps': f'group{kpx_group}_power_proxy_ldaps'})

    # 7) 풍향
    wd_rad = np.arctan2(-out['heightAboveGround_10_10u'], -out['heightAboveGround_10_10v'])
    out['wd10_sin_ldaps'] = np.sin(wd_rad)
    out['wd10_cos_ldaps'] = np.cos(wd_rad)

    # 8) 누수 검증용 시각
    avail = ldaps.groupby('forecast_kst_dtm')['data_available_kst_dtm'].first()
    out = out.join(avail).reset_index()

    # 9) 시간 파생변수
    dtm = pd.to_datetime(out['forecast_kst_dtm'])
    out['hour'] = dtm.dt.hour
    out['month'] = dtm.dt.month

    return out

In [8]:
# GFS
def preprocess_gfs_group(
        gfs_path: str,
        kpx_group: int,
        use_grids = (1, 2, 4, 5),
        cp: float = 0.35,
        info_path: str = "/Users/hyukjin/Documents/Gongmo/BARAM-2026/data/info_include_lat_lon.csv"
) -> pd.DataFrame:
    REF_T = 2.0
    LAPSE_RATE, G, R_D = 0.0065, 9.80665, 287.05

    gfs = pd.read_csv(gfs_path, encoding='utf-8-sig')
    info = pd.read_csv(info_path, encoding='utf-8-sig')
    group_info = info[info['KPX그룹'] == kpx_group].copy()
    hub_height = float(group_info['Hub Height(m)'].iloc[0])
    rotor_d    = float(group_info['Rotor Diameter(m)'].iloc[0])
    swept_area = np.pi * (rotor_d / 2) ** 2
    lat_c = group_info['latitude'].mean()   # 터빈 군집 중심
    lon_c = group_info['longitude'].mean()

    # 0) 결측 보간 (배치 내) — 안전장치
    gfs = gfs.sort_values(['grid_id', 'data_available_kst_dtm', 'forecast_kst_dtm']).reset_index(drop=True)
    num_cols = [c for c in gfs.select_dtypes(include=[np.number]).columns if c != 'grid_id']
    gfs[num_cols] = gfs.groupby(['grid_id', 'data_available_kst_dtm'])[num_cols].transform(
        lambda g: g.interpolate(method='linear', limit_direction='both'))

    # 1) 감싸는 격자만
    gfs = gfs[gfs['grid_id'].isin(use_grids)].copy()

    # 2) 격자별 파생
    gfs['gfs_ws10']  = np.hypot(gfs['heightAboveGround_10_10u'],   gfs['heightAboveGround_10_10v'])
    gfs['gfs_ws80']  = np.hypot(gfs['heightAboveGround_80_u'],     gfs['heightAboveGround_80_v'])
    gfs['gfs_ws100'] = np.hypot(gfs['heightAboveGround_100_100u'], gfs['heightAboveGround_100_100v'])

    a = np.log(np.clip(gfs['gfs_ws100'], 0.5, None) / np.clip(gfs['gfs_ws80'], 0.5, None)) / np.log(100/80)
    a = a.clip(0.05, 0.4)
    gfs['gfs_alpha'] = a
    gfs['gfs_ws_hub'] = gfs['gfs_ws100'] * (hub_height / 100.0) ** a

    # 상층 종관풍 (GFS 고유)
    gfs['gfs_ws850']  = np.hypot(gfs['isobaricInhPa_850_u'], gfs['isobaricInhPa_850_v'])
    gfs['gfs_ws700']  = np.hypot(gfs['isobaricInhPa_700_u'], gfs['isobaricInhPa_700_v'])
    gfs['gfs_ws_pbl'] = np.hypot(gfs['planetaryBoundaryLayer_0_u'], gfs['planetaryBoundaryLayer_0_v'])

    # 공기밀도 (GFS 2t/2sh/sp)
    gfs['gfs_t_hub'] = gfs['heightAboveGround_2_2t'] - LAPSE_RATE * (hub_height - REF_T)
    q = gfs['heightAboveGround_2_2sh']
    tv_sfc = gfs['heightAboveGround_2_2t'] * (1 + 0.61 * q)
    tv_hub = gfs['gfs_t_hub'] * (1 + 0.61 * q)
    tv_mean = (tv_sfc + tv_hub) / 2
    p_hub = gfs['surface_0_sp'] * np.exp(-G * hub_height / (R_D * tv_mean))
    gfs['gfs_rho_hub'] = p_hub / (R_D * tv_hub)

    gfs['gfs_power_proxy'] = 0.5 * gfs['gfs_rho_hub'] * swept_area * gfs['gfs_ws_hub'] ** 3 * cp / 1000

    # 3) centroid IDW 가중치
    def haversine_km(lat1, lon1, lat2, lon2):
        R = 6371.0088
        lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
        a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
        return 2 * R * np.arcsin(np.sqrt(a))

    gd = gfs.drop_duplicates('grid_id')[['grid_id', 'latitude', 'longitude']]
    dist = {int(r.grid_id): haversine_km(lat_c, lon_c, r.latitude, r.longitude) for _, r in gd.iterrows()}
    inv = {g: 1/dd for g, dd in dist.items()}
    s = sum(inv.values())
    wmap = {g: v/s for g, v in inv.items()}
    gfs['w'] = gfs['grid_id'].map(wmap)

    # 4) centroid 1점으로 IDW 집계
    AGG_COLS = [
        'gfs_power_proxy',
        'gfs_ws_hub', 'gfs_ws100', 'gfs_ws80', 'gfs_ws10', 'gfs_alpha',
        'gfs_ws850', 'gfs_ws700', 'gfs_ws_pbl',
        'gfs_rho_hub', 'gfs_t_hub',
        'surface_0_gust', 'planetaryBoundaryLayer_0_VRATE',
        'isobaricInhPa_500_gh', 'isobaricInhPa_850_t', 'atmosphere_0_tcc',
        # 풍향 계산용 성분 (100m=허브 근접, 850=종관)
        'heightAboveGround_100_100u', 'heightAboveGround_100_100v',
        'isobaricInhPa_850_u', 'isobaricInhPa_850_v',
    ]
    for c in AGG_COLS:
        gfs[c] = gfs[c] * gfs['w']
    out = gfs.groupby('forecast_kst_dtm')[AGG_COLS].sum()

    # 5) 풍향 sin/cos (100m 허브풍 + 850hPa 종관풍)
    wd100 = np.arctan2(-out['heightAboveGround_100_100u'], -out['heightAboveGround_100_100v'])
    out['gfs_wd100_sin'] = np.sin(wd100)
    out['gfs_wd100_cos'] = np.cos(wd100)
    wd850 = np.arctan2(-out['isobaricInhPa_850_u'], -out['isobaricInhPa_850_v'])
    out['gfs_wd850_sin'] = np.sin(wd850)
    out['gfs_wd850_cos'] = np.cos(wd850)
    out = out.drop(columns=['heightAboveGround_100_100u', 'heightAboveGround_100_100v',
                            'isobaricInhPa_850_u', 'isobaricInhPa_850_v'])

    # 6) 누수 검증용 시각
    avail = gfs.groupby('forecast_kst_dtm')['data_available_kst_dtm'].first()
    out = out.join(avail).reset_index()
    return out